# Single-Target Tutorial

This notebook builds a simple single-target passive-sonar scenario step by step, demonstrating each major component within the plugin.

We will use the following components in this tutorial:
- Platform: Passive towed linear array
- Propagation: Cylindrical
- Signals: Broadband ship source & coloured ambient background noise
- Detection: Cell-averaging constant false alarm rate (CA-CFAR) & peak picking
- Tracking: Single target Kalman filter (KF) with probabilistic data association (PDA)

We will first define the global simulation parameters that specify the duration of the
simulation and the simulation time interval.

The simulation time interval is also equivalent to the signal processing integration
rate.

In [ ]:
from datetime import datetime, timedelta

import numpy as np

# Random seed for reproducibility
seed = 2000
np.random.seed(seed)

# Simulation parameters
sim_duration = timedelta(seconds=900)
time_interval = timedelta(seconds=5)

num_steps = int(sim_duration.total_seconds() / time_interval.total_seconds())
start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
timesteps = [start_time + i * time_interval for i in range(num_steps)]

## Platform Setup and Generation

The platform is the vehicle that houses the sensor system. In this case, we will
consider the platform to be a frigate, and we will model the sonar system as a towed
array of hydrophones. The system receives and processes signals, but does not emit any
signals of its own, hence it is a passive system.

The dynamics of the platform and towed array are modelled simply in a
"follow-the-leader" manner. Specifically, the platform and each hydrophone is a 3D point
in space, the "leader" is the platform and controls the trajectory of all "follower"
points. Therefore the position of each hydrophone is determined by the position of the
hydrophone ahead of it one time step previously.

This approach allows the curvature of the towed array to be approximately modelled,
enabling downstream influence on the beamforming process.

The key component here is the `TowedArrayPlatform`, which inherits from Stone Soup's
`MultiTransitionMovingPlatform`, allowing different transition models to be used at
various user-specified times. In addition, `TowedArrayPlatform` requires arguments for
the number of array sensors, the length of the tow cable, the spacing between adjacent
sensors, and the depth at which the array is operating at.

Note that the tow cable is the length between the plaform (ship) and the first sensor
in the array.

For this tutorial, let us consider an array consisting of 50 hydrophones with a spacing
of 0.5 m, operating at a depth of 50 m. The ship towing the array, which is separated
by a 100 m cable, will be on the sea-surface. The platform will follow a constant-velocity
model, travelling in a straight line with no deviation in course or speed.

In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthState

from nereus.platform import TowedArrayPlatform

# Define the platform's initial state and transition model
platform_start_vector = np.array([-2000.0, 5.0, 2000.0, 0.0, -5.0, 0.0])
platform_position_mapping = [0, 2, 4]
platform_velocity_mapping = [1, 3, 5]
platform_transition_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)
])

# Define the towed array parameters
num_sensors = 50
tow_cable_length_m = 100.0
sensor_spacing_m = 0.5
array_depth_m = -50.0

# Create the towed array platform and simulate its movement over time
platform_initial_state = GroundTruthState(platform_start_vector, timestamp=start_time)
platform = TowedArrayPlatform(
    states=platform_initial_state,
    position_mapping=platform_position_mapping,
    velocity_mapping=platform_velocity_mapping,
    transition_models=[platform_transition_model],
    transition_times=[sim_duration],
    num_sensors=num_sensors,
    cable_length_m=tow_cable_length_m,
    sensor_spacing_m=sensor_spacing_m,
    array_depth_m=array_depth_m,
)

for timestamp in timesteps[1:]:
    platform.move(timestamp)

## Ground Truth Setup and Generation

Next we will create the target ground truth. The target states simply use Stone Soup's
`GroundTruthState` class, but we inject additional metadata associated with the target's
source signal. 

The source signal is defined as a broadband source with multiple tonal components
defined over a discrete set of frequencies. Tonal frequencies are approximate
representations of distinct and prominent on-board contributions to the signal. Each
tonal frequency has an amplitude and phase. The width of the tonal frequency is defined
by its bandwidth. Unmodelled contributions to the signal are represented by a stochastic
noise process, e.g., pink noise.

In [ ]:
import numpy as np
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.plotter import plot_world

# Define the target's initial state and transition model
target_start_vector = np.array([0.0, 0.0, 0.0, 8.0, -5.0, 0.0])
target_transition_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)
])
target_position_mapping = [0, 2, 4]
target_velocity_mapping = [1, 3, 5]

# Define the target's signal parameters
target_amplitudes_upa = 10 ** (np.random.uniform(97, 112, 4) / 20)
target_frequencies_hz = np.random.uniform(25.0, 200.0, 4)
target_phases_rad = np.random.uniform(0, 2 * np.pi, 4)
target_tonal_bandwidth_hz = np.random.uniform(0.5, 2.0)
target_noise_amplitude_upa = 10 ** (90 / 20)
target_noise_spectral_exponent = -1.0 # Pink noise

target_metadata = {
    'amplitudes_upa': target_amplitudes_upa,
    'frequencies_hz': target_frequencies_hz,
    'phases_rad': target_phases_rad,
    'position_mapping': target_position_mapping,
    'velocity_mapping': target_velocity_mapping,
    'tonal_bandwidth_hz': target_tonal_bandwidth_hz,
    'noise_amplitude_upa': target_noise_amplitude_upa,
    'noise_spectral_exponent': target_noise_spectral_exponent,
}

# Simulate the target's movement over time and create a ground truth path
target_states = [
    GroundTruthState(
        target_start_vector, timestamp=start_time, metadata=target_metadata
    )
]
for timestamp in timesteps[1:]:
    dt = timestamp - target_states[-1].timestamp
    new_state_vector = target_transition_model.function(
        target_states[-1], noise=False, time_interval=dt
    )
    target_states.append(
        GroundTruthState(
            new_state_vector, timestamp=timestamp, metadata=target_metadata
        )
    )

target_truth = GroundTruthPath(target_states)
target_truths = [target_truth]

plot_world(truths=target_truths, platform=platform).show()

## Propagation Model

Create the acoustic environment and cylindrical propagation model.



In [ ]:
from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import CylindricalAcousticPropagationModel

ssp = Linear(surface_speed=1500.0, gradient=0.2)
bathymetry = FlatBathymetry(depth=-150.0)
attenuation_factor = 0.5

propagation_model = CylindricalAcousticPropagationModel(
    ssp=ssp,
    attenuation_factor=attenuation_factor,
)

## Signal Model

We now define the acoustic content that reaches the array: a target source model and
an ambient background model. Both are configured to be compatible with the same
sampling and processing settings used by the simulator.

First, we set the core signal-processing parameters. `sampling_rate_hz` fixes the time
resolution and Nyquist limit, while `frame_len` and `hop_factor` control the short-time
processing window and overlap used in downstream broadband beamforming.

Ambient noise is represented by `ColouredNoise`, with amplitude in micro-Pascal and a
spectral exponent. Using an exponent of `-1` gives pink-noise-like behaviour, which is a
useful first-order approximation for ocean background in this tutorial. The ambient model
duration is set to one processing interval (`time_interval.total_seconds()`), so new
noise is generated each interval.

The target source is generated with `BroadbandShipSignal`. This combines tonal and
broadband stochastic components using metadata defined in the ground-truth section
(tonal bandwidth plus noise amplitude/spectral slope). Its duration spans the full
simulation (`duration_s`) so the source remains available throughout all timesteps.

Finally, `noise_freq_range_hz=(0, sampling_rate_hz/2)` limits stochastic source noise to
the physically valid one-sided frequency range, and setting
`tonal_noise_is_constant=True` and `noise_is_constant=True` keeps source characteristics
fixed over time. This helps isolate the effects of propagation, platform motion, and
beamforming in later sections.


In [ ]:
from nereus.signal.ambient import ColouredNoise
from nereus.signal.anthropogenic import BroadbandShipSignal

sampling_rate_hz = 500.0
frame_len = 500
hop_factor = 2
fade_in_ms = 1000.0
duration_s = num_steps * time_interval.total_seconds()

ambient_amplitude_upa = 10 ** (45 / 20)
ambient_spectral_exponent = -1
ambient_noise_model = ColouredNoise(
    amplitude_upa=ambient_amplitude_upa,
    spectral_exponent=ambient_spectral_exponent,
    duration_s=time_interval.total_seconds(),
    sampling_rate_hz=sampling_rate_hz,
)

signal_model = BroadbandShipSignal(
    duration_s=duration_s,
    sampling_rate_hz=sampling_rate_hz,
    frame_len=frame_len,
    hop_factor=hop_factor,
    tonal_bandwidth_hz=target_tonal_bandwidth_hz,
    noise_amplitude_upa=target_noise_amplitude_upa,
    noise_spectral_exponent=target_noise_spectral_exponent,
    noise_freq_range_hz=(0.0, sampling_rate_hz / 2),
    tonal_noise_is_constant=True,
    noise_is_constant=True,
)

## Beamformer

This stage forms spatial energy across azimuth and runs the detection chain.

We configure an MVDR beamformer and compute steering vectors over a full
`[-pi, pi]` azimuth grid. Processing is restricted to a frequency band
(`fmin` to `fmax`) where target energy is expected.

`BroadbandPassiveSonarArraySimulator` then combines platform geometry, propagation,
source/noise models, and steering to generate beamformed snapshots through time.

Detections are extracted with CA-CFAR and optionally refined by peak-picking to reduce
clustered threshold crossings. We finally inspect the bearing-time record (BTR) both with
and without overlaid detections.


In [ ]:
import numpy as np

from nereus.detector import CACFARDetector, PassiveSonarDetector, PeakDetector
from nereus.plotter import plot_btr
from nereus.sigproc import (
    MinimumVarianceDistortionlessResponseBeamformer,
    SteeringCalculator,
)
from nereus.simulator import BroadbandPassiveSonarArraySimulator

shading = None
beamforming_domain = 'broadband_power'
steering_azimuths_rad = np.linspace(-np.pi, np.pi, 181)
fmin = 100.0
fmax = 125.0

beamformer = MinimumVarianceDistortionlessResponseBeamformer(
    sampling_rate_hz=sampling_rate_hz,
    fmin=fmin,
    fmax=fmax,
)

steering_calculator = SteeringCalculator(
    ssp=ssp,
    steering_azimuths_rad=steering_azimuths_rad,
)

simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=propagation_model,
    signal_models=[signal_model],
    noise_model=ambient_noise_model,
    beamformer=beamformer,
    steering_calculator=steering_calculator,
    ground_truth_paths=target_truths,
    fade_in_ms=fade_in_ms,
)

num_guard_cells = 2
num_training_cells = 16
threshold_factor = 1.5
peak_distance = 3

cfar_detector = CACFARDetector(
    num_guard_cells=num_guard_cells,
    num_training_cells=num_training_cells,
    threshold_factor=threshold_factor,
)
detection_chain = [cfar_detector]
if peak_distance > 0:
    detection_chain.append(PeakDetector(distance=peak_distance))

detector = PassiveSonarDetector(
    detection_chain=detection_chain,
    sensor_data_gen=simulator.sensor_data_gen(),
    steering_azimuths_rad=steering_azimuths_rad,
)

all_detections = list(detector.detections_gen(progress_bar=True))
snr_map = detector.snr_history

detections_for_plotter = [d for _, detections in all_detections for d in detections]

print(f"Total no. of detections: {len(detections_for_plotter)}")

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, subplot_titles=("SNR Map", "SNR Map with Detections")
)

plot_btr(
    data=snr_map,
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
    fig=fig, row=1, col=1
)
plot_btr(
    data=snr_map,
    detections=detections_for_plotter,
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
    fig=fig, row=1, col=2
)

fig.update_layout(width=1200, height=700, yaxis2=dict(title=""))
fig.show()

## Tracker

Finally, we estimate a continuous bearing track from the discrete detections.

The tracker state is `[bearing, bearing_rate]`, propagated by a constant-velocity
transition model and updated with a linear Gaussian bearing measurement model.

Data association is handled by PDA: all feasible detection-to-track hypotheses are
weighted probabilistically each step. The Gaussian mixture of posterior hypotheses is then
collapsed to a single Gaussian estimate before appending to the
track.

To evaluate performance on the same BTR axes, we also convert Cartesian target truth to
relative bearing truth with respect to the array reference position at each timestamp,
then plot truth, detections, and the final track together.


In [ ]:
bearing_states = []
for target_state in target_truth:
    platform_state = platform.get_platform_state_at(target_state.timestamp)
    ref_sensor_position = np.mean(platform_state.array.state_vector, axis=1)
    target_xy = np.array([target_state.state_vector[0], target_state.state_vector[2]])
    relative_position = target_xy - ref_sensor_position[:2]
    bearing = np.arctan2(relative_position[1], relative_position[0])
    bearing_states.append(
        GroundTruthState(np.array([bearing]), timestamp=target_state.timestamp)
    )

relative_bearing_truth = GroundTruthPath(bearing_states)
relative_bearing_truths = [relative_bearing_truth]

In [ ]:
import numpy as np
from stonesoup.dataassociator.probability import PDA
from stonesoup.deleter.time import UpdateTimeStepsDeleter
from stonesoup.functions import mod_bearing
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.initiator.simple import SinglePointInitiator
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.tracker.simple import SingleTargetMixtureTracker
from stonesoup.types.state import GaussianState
from stonesoup.types.track import Track
from stonesoup.updater.kalman import KalmanUpdater

from nereus.plotter import plot_btr

predictor = KalmanPredictor(ConstantVelocity(0.000001))

measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(1) ** 2]]),
)

updater = KalmanUpdater(measurement_model=measurement_model)

hypothesiser = PDAHypothesiser(
    predictor=predictor,
    updater=updater,
    clutter_spatial_density=5 / np.pi,
    prob_detect=0.95,
)
data_associator = PDA(hypothesiser=hypothesiser)

initial_bearing = float(relative_bearing_truth[0].state_vector[0])

prior_state = GaussianState(
    np.array([initial_bearing, 0.0]),
    np.diag([np.deg2rad(5) ** 2, np.deg2rad(0.5) ** 2]),
    timestamp=start_time,
)

initiator = SinglePointInitiator(
    prior_state=prior_state,
    measurement_model=measurement_model,
    updater=updater,
)

deleter = UpdateTimeStepsDeleter(time_steps_since_update=99999)

kf = SingleTargetMixtureTracker(
    initiator=initiator,
    deleter=deleter,
    detector=all_detections,
    data_associator=data_associator,
    updater=updater,
)

seed_track = Track(states=[prior_state])
kf._track = seed_track

tracks = set()

for _, current_tracks in kf:
    for track in current_tracks:
        track[-1].state_vector[0, 0] = mod_bearing(float(track[-1].state_vector[0, 0]))
    tracks |= current_tracks

plot_btr(
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
    truths=relative_bearing_truths,
    detections=detections_for_plotter,
    tracks=tracks,
    figsize=(700, 700),
).show()